In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta
from apu_method import apu_uncertainty_lstm

# === Funções auxiliares ===
def corr_upper_from_samples(X):
    R = np.corrcoef(X, rowvar=False)
    R = np.clip(R, -1.0, 1.0)
    np.fill_diagonal(R, 1.0)
    return np.triu(R, k=0)

def uncertainty_scale_factor(scaler):
    if hasattr(scaler, "data_max_") and hasattr(scaler, "data_min_"):
        return float(scaler.data_max_[0] - scaler.data_min_[0])
    if hasattr(scaler, "scale_"):
        return float(scaler.scale_[0])
    return 1.0

def cria_janelas(series, janela=10, atraso=48):
    X, y = [], []
    for i in range(len(series) - janela - atraso):
        X.append(series[i:i+janela])
        y.append(series[i+janela+atraso-1])
    return np.array(X), np.array(y)

# === CONFIGURAÇÕES GERAIS ===
u_x_seq_cm = np.array([5.0]*10)   # incerteza de cada entrada em cm
u_y_cm = 5.0                      # incerteza da saída real (nível medido)
k_cov = 2.0                       # fator de cobertura (95%)
janela = 10

# === CARREGA SÉRIE DE TESTE ===
df_teste = pd.read_excel(r"C:\Users\AdmPDI\Documents\Doutorado\Dados\Projeto_1\teste.xlsx")
df_teste['datetime'] = pd.to_datetime(df_teste['datetime'])
df_teste['nivel_cm'] = pd.to_numeric(df_teste['nivel_cm'], errors='coerce')
serie = df_teste.dropna().set_index('datetime')
serie_vals = serie[['nivel_cm']].interpolate(method='linear').fillna(method='ffill').values

# === MODELOS POR HORIZONTE ===
horizontes = {120: 8, 240: 16, 480: 32, 720: 48, 1440: 96, 2880: 192}
resultados = []

for minutos, atraso in horizontes.items():
    print(f"\n▶️ Processando modelo para {minutos} minutos...")

    # Carrega modelo e scaler
    model = tf.keras.models.load_model(f"modelo_previsao_{minutos}min.h5", compile=False)
    scaler = joblib.load(f"scaler_previsao_{minutos}min.save")

    # Gera pares X, y a partir da série de teste
    X_raw, y_raw = cria_janelas(serie_vals, janela=janela, atraso=atraso)
    X_flat = X_raw.reshape(len(X_raw), janela)
    y_raw = y_raw.reshape(-1, 1)

    # Escala
    X_scaled = scaler.transform(X_flat.reshape(-1, 1)).reshape(X_flat.shape)
    y_scaled = scaler.transform(y_raw)

    # Correlação entre atrasos
    M3 = corr_upper_from_samples(X_scaled)

    # Resíduos para SE/RE
    X_model_in = X_scaled.reshape(len(X_scaled), janela, 1)
    y_pred_scaled = model.predict(X_model_in, verbose=0)
    y_pred_cm = scaler.inverse_transform(y_pred_scaled)
    y_cm = scaler.inverse_transform(y_scaled)
    residuos_cm = (y_pred_cm.flatten() - y_cm.flatten())
    SE_cm = float(np.mean(residuos_cm))
    RE_cm = float(np.std(residuos_cm - SE_cm))

    # Escalas
    s = uncertainty_scale_factor(scaler)
    u_x_seq_scaled = u_x_seq_cm / s
    u_y_scaled = u_y_cm / s
    SE_scaled = SE_cm / s
    RE_scaled = RE_cm / s

    # Previsão APU para cada amostra do teste
    for i in range(X_scaled.shape[0]):
        x_seq_scaled = X_scaled[i].reshape(1, janela, 1)

        # Corr simétrica
        R_corr = M3.copy()
        for r in range(R_corr.shape[0]):
            for c in range(r):
                R_corr[r, c] = R_corr[c, r]

        out = apu_uncertainty_lstm(
            model=model,
            x_seq=x_seq_scaled.reshape(-1),
            u_x_seq=u_x_seq_scaled,
            SE=SE_scaled,
            RE=RE_scaled,
            u_y=u_y_scaled,
            k_coverage=k_cov,
            rho=R_corr
        )

        y_pred_cm_single = scaler.inverse_transform([[out['y_pred']]])[0, 0]
        u_D_cm = out['u_D'] * s
        u_M_cm = out['u_M'] * s
        u_P_cm = out['u_P'] * s
        U_cm   = out['U'] * s

        t_prev = serie.index[i + janela + atraso - 1]  # timestamp correspondente

        resultados.append({
            'Horizonte (min)': minutos,
            'Data prevista': t_prev,
            'Previsão (cm)': y_pred_cm_single,
            'Real (cm)': serie.loc[t_prev, 'nivel_cm'],
            'u_D (dados) [cm]': u_D_cm,
            'u_M (modelo) [cm]': u_M_cm,
            'u_P (combinada) [cm]': u_P_cm,
            'U (expandida) [cm]': U_cm,
            'Previsão ± U': f"{y_pred_cm_single:.2f} ± {U_cm:.2f} cm"
        })

# === RESULTADO FINAL ===
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(["Horizonte (min)", "Data prevista"]).reset_index(drop=True)

# Exibe os primeiros resultados
print(df_resultados.head())


C:\Users\AdmPDI\AppData\Local\Temp\ipykernel_25868\1376359266.py:40: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  serie_vals = serie[['nivel_cm']].interpolate(method='linear').fillna(method='ffill').values



▶️ Processando modelo para 120 minutos...

▶️ Processando modelo para 240 minutos...

▶️ Processando modelo para 480 minutos...

▶️ Processando modelo para 720 minutos...

▶️ Processando modelo para 1440 minutos...

▶️ Processando modelo para 2880 minutos...
   Horizonte (min)       Data prevista  Previsão (cm)  Real (cm)  \
0              120 2025-06-03 13:30:00     342.476815        343   
1              120 2025-06-03 13:45:00     342.531278        342   
2              120 2025-06-03 14:00:00     342.612200        342   
3              120 2025-06-03 14:15:00     341.252891        342   
4              120 2025-06-03 14:30:00     341.389634        342   

   u_D (dados) [cm]  u_M (modelo) [cm]  u_P (combinada) [cm]  \
0          7.033868           4.477615              8.338125   
1          7.033736           4.477615              8.338014   
2          7.033656           4.477615              8.337947   
3          7.042078           4.477615              8.345053   
4          

In [7]:
import os

# Caminho da pasta onde deseja salvar
pasta_destino = r"C:\Users\AdmPDI\Documents\Doutorado\Dados\Projeto_1"

# Caminho completo do arquivo de saída
caminho_saida = os.path.join(pasta_destino, "resultados_apu_teste.xlsx")

# Salva o DataFrame
df_resultados.to_excel(caminho_saida, index=False)

print("Arquivo salvo em:", caminho_saida)



Arquivo salvo em: C:\Users\AdmPDI\Documents\Doutorado\Dados\Projeto_1\resultados_apu_teste.xlsx


In [14]:
# Caminho completo
caminho = r"C:\Users\AdmPDI\Documents\Doutorado\Dados\Projeto_1\resultados_apu_teste.xlsx"

# Leitura do Excel
df_resultados = pd.read_excel(caminho, engine="openpyxl")

# Exibe as primeiras linhas
print(df_resultados.head())


   Horizonte (min)       Data prevista  Previsão (cm)  Real (cm)  \
0              120 2025-06-03 13:30:00     342.476815        343   
1              120 2025-06-03 13:45:00     342.531278        342   
2              120 2025-06-03 14:00:00     342.612200        342   
3              120 2025-06-03 14:15:00     341.252891        342   
4              120 2025-06-03 14:30:00     341.389634        342   

   u_D (dados) [cm]  u_M (modelo) [cm]  u_P (combinada) [cm]  \
0          7.033868           4.477615              8.338125   
1          7.033736           4.477615              8.338014   
2          7.033656           4.477615              8.337947   
3          7.042078           4.477615              8.345053   
4          7.036435           4.477615              8.340292   

   U (expandida) [cm]       Previsão ± U  
0           15.192611  342.48 ± 15.19 cm  
1           15.192389  342.53 ± 15.19 cm  
2           15.192254  342.61 ± 15.19 cm  
3           15.206466  341.25 ± 15

In [15]:
import plotly.graph_objects as go

# === 2. Gera um gráfico interativo para cada horizonte de previsão ===
def plot_interativo_apu(df_resultados):
    horizontes = sorted(df_resultados["Horizonte (min)"].unique())

    for horizonte in horizontes:
        df_h = df_resultados[df_resultados["Horizonte (min)"] == horizonte]

        fig = go.Figure()

        # Linha do valor real
        fig.add_trace(go.Scatter(
            x=df_h["Data prevista"],
            y=df_h["Real (cm)"],
            mode='lines',
            name='Real',
            line=dict(color='black')
        ))

        # Linha da previsão
        fig.add_trace(go.Scatter(
            x=df_h["Data prevista"],
            y=df_h["Previsão (cm)"],
            mode='lines',
            name='Previsão',
            line=dict(color='red')
        ))

        # Faixa superior (limite superior da incerteza)
        fig.add_trace(go.Scatter(
            x=df_h["Data prevista"],
            y=df_h["Previsão (cm)"] + df_h["U (expandida) [cm]"],
            mode='lines',
            line=dict(width=0),
            showlegend=False
        ))

        # Faixa inferior + preenchimento
        fig.add_trace(go.Scatter(
            x=df_h["Data prevista"],
            y=df_h["Previsão (cm)"] - df_h["U (expandida) [cm]"],
            mode='lines',
            fill='tonexty',
            fillcolor='rgba(173,216,230,0.3)',  # azul claro
            line=dict(width=0),
            name='Incerteza ±U'
        ))

        # Layout do gráfico
        fig.update_layout(
            title=f"Previsão vs Real com Incerteza Expandida (Horizonte: {horizonte} min)",
            xaxis_title="Data Prevista",
            yaxis_title="Nível do Rio (cm)",
            legend=dict(x=0.01, y=0.99),
            template="plotly_white"
        )

        fig.show()

# === 3. Executa a função de plotagem ===
plot_interativo_apu(df_resultados)



ERROR:tensorflow:==================================
Object was never used (type <class 'tensorflow.python.ops.tensor_array_ops.TensorArray'>):
If you want to mark it as used call its "mark_used()" method.
It was originally created here:
  File "c:\Users\AdmPDI\Documents\Conda3\envs\riosinos\Lib\site-packages\keras\src\backend\tensorflow\rnn.py", line 418, in <genexpr>
    output_ta_t = tuple(  File "c:\Users\AdmPDI\Documents\Conda3\envs\riosinos\Lib\site-packages\tensorflow\python\util\tf_should_use.py", line 288, in wrapped
    return _add_should_use_warning(fn(*args, **kwargs),
